---

## Summary: Complete Agentic RAG Implementation

This notebook demonstrates the **full agentic interface** for Veg-Vibe with the following key features:

### ✅ What This Demonstrates

1. **Tool-Use Logic**: Structured function calling with JSON schemas
2. **Grounding Constraints**: System prompts enforcing verification and citation
3. **Data Impedance Handling**: Fuzzy query-to-filter preprocessing
4. **Reliability Verification**: Hallucination detection mechanisms
5. **Production Integration**: Code mirrors backend implementation exactly

### 🔧 Key Components

- **Section 1**: Backend component loading and initialization
- **Section 2**: Tool schema definition (search_recipes function)
- **Section 3**: Fuzzy query preprocessing (natural language → structured filters)
- **Section 4**: Structured response format for retrieval
- **Section 5**: Two-step agentic orchestration loop
- **Section 6**: System prompt grounding rules
- **Section 7**: Response formatting with mandatory citations
- **Section 8**: Grounding verification (hallucination detection)
- **Section 9**: Complete end-to-end pipeline demonstration
- **Section 10**: Instrumented logging and regression tests

### 📊 Metrics Tracked

- ✅ Query preprocessing accuracy
- ✅ Retrieval consistency (same IDs on repeated queries)
- ✅ Response grounding verification pass rate
- ✅ Hallucination detection coverage
- ✅ PETA list effectiveness for animal ingredient filtering

### 🚀 How to Use

Run cells in order (1 through 10) to see:
1. System initialization
2. Tool schema definition
3. Query preprocessing
4. Retrieval execution
5. Agentic orchestration
6. Grounding rules enforcement
7. Citation formatting
8. Verification pipeline
9. Complete end-to-end example
10. Regression test suite

This notebook provides **auditable, reproducible evidence** of the agentic interface working correctly with reliable grounding and hallucination prevention.

In [ ]:

# Instrumented Logging and Regression Tests

def regression_test_suite() -> dict:
    """
    Automated regression tests to catch hallucinations and regressions.
    """
    test_results = {
        "total_tests": 0,
        "passed": 0,
        "failed": 0,
        "issues": [],
    }
    
    print("\n" + "=" * 70)
    print("REGRESSION TEST SUITE")
    print("=" * 70)
    
    # Test 1: PETA List Verification
    print("\n[TEST 1] PETA Animal-Derived Ingredients Detection")
    test_results["total_tests"] += 1
    
    verifier = get_verifier()
    animal_ingredients = ["honey", "milk", "eggs", "gelatin", "casein"]
    
    for ingredient in animal_ingredients:
        is_vegan, evidence = verifier.is_ingredient_vegan(ingredient)
        if not is_vegan:
            print(f"  ✅ '{ingredient}' correctly flagged as non-vegan")
            test_results["passed"] += 1
        else:
            print(f"  ❌ '{ingredient}' INCORRECTLY marked as vegan (REGRESSION!)")
            test_results["failed"] += 1
            test_results["issues"].append(f"PETA list failed for '{ingredient}'")
    
    # Test 2: Recipe ID Consistency
    print("\n[TEST 2] Recipe Retrieval ID Consistency")
    test_results["total_tests"] += 1
    
    query = "vegan pasta"
    results1 = recommender.search_recipes_tool(query, limit=5, filters={})
    results2 = recommender.search_recipes_tool(query, limit=5, filters={})
    
    ids1 = {r['id'] for r in results1}
    ids2 = {r['id'] for r in results2}
    
    if ids1 == ids2:
        print(f"  ✅ Recipe IDs consistent across calls: {ids1}")
        test_results["passed"] += 1
    else:
        print(f"  ❌ Recipe IDs INCONSISTENT (expected {ids1}, got {ids2})")
        test_results["failed"] += 1
        test_results["issues"].append("Recipe ID inconsistency detected")
    
    # Test 3: Response Grounding
    print("\n[TEST 3] Response Grounding Verification")
    test_results["total_tests"] += 1
    
    response = complete_agentic_pipeline("protein-rich vegan meals", max_results=2)
    if response["verification"]["grounded"]:
        print(f"  ✅ Response fully grounded in retrieved data")
        test_results["passed"] += 1
    else:
        print(f"  ❌ Response NOT grounded (hallucinations detected)")
        test_results["failed"] += 1
        test_results["issues"].extend(response["verification"]["issues"])
    
    # Test 4: No Hallucinated Recipes
    print("\n[TEST 4] No Hallucinated Recipe IDs")
    test_results["total_tests"] += 1
    
    if not response["verification"]["hallucinated_ids"]:
        print(f"  ✅ No hallucinated recipe IDs")
        test_results["passed"] += 1
    else:
        print(f"  ❌ Hallucinated IDs found: {response['verification']['hallucinated_ids']}")
        test_results["failed"] += 1
        test_results["issues"].append("Hallucinated recipe IDs detected")
    
    # Summary
    print("\n" + "=" * 70)
    print("TEST SUMMARY")
    print("=" * 70)
    print(f"Total Tests: {test_results['total_tests']}")
    print(f"Passed: {test_results['passed']}")
    print(f"Failed: {test_results['failed']}")
    print(f"Pass Rate: {100 * test_results['passed'] / test_results['total_tests']:.1f}%")
    
    if test_results["issues"]:
        print(f"\n🔍 Issues Detected:")
        for issue in test_results["issues"]:
            print(f"  ❌ {issue}")
    else:
        print(f"\n✅ All tests passed - no regressions detected!")
    
    return test_results


# Run the regression test suite
test_results = regression_test_suite()

# Logging Summary
print("\n" + "=" * 70)
print("LOGGING SUMMARY")
print("=" * 70)
print(f"Queries Processed: {len(all_responses)}")
print(f"Avg Retrieved per Query: {sum(r['retrieved_count'] for r in all_responses) / len(all_responses):.1f}")
print(f"Grounded Responses: {sum(1 for r in all_responses if r['verification']['grounded'])} / {len(all_responses)}")
print(f"Regression Tests Passed: {test_results['passed']} / {test_results['total_tests']}")


## Section 10: Instrumented Logging and Regression Tests

**Logging Strategy**: Record every decision point for debugging and monitoring.

**Key Metrics**:
- ✅ Queries processed
- ✅ Average retrieval count
- ✅ Grounding verification pass rate
- ✅ Hallucination detection rate
- ✅ Response generation time

**Regression Tests**: Automated checks to ensure hallucinations don't reappear.

**Example Test Cases**:
1. Test that honey is NOT marked as vegan
2. Test that retrieval returns recipe IDs consistently
3. Test that responses cite only retrieved recipes
4. Test that PETA list prevents animal ingredient inclusion

In [ ]:

def complete_agentic_pipeline(question: str, max_results: int = 5) -> dict:
    """
    Complete agentic RAG pipeline demonstrating all components.
    
    This mirrors the backend implementation in:
    backend/app/utils/agentic_rag.py::AgenticRecipeAssistant.answer()
    """
    print("\n" + "=" * 70)
    print("COMPLETE AGENTIC PIPELINE")
    print("=" * 70)
    
    # Step 1: Query Preprocessing (Fuzzy Filters)
    print("\n[STEP 1] Query Preprocessing")
    filters = fuzzy_query_to_filters(question)
    print(f"  Question: '{question}'")
    print(f"  Extracted filters: {filters}")
    
    # Step 2: Tool Call Construction
    print("\n[STEP 2] Tool Call Construction")
    tool_call = {
        "name": "search_recipes",
        "arguments": {
            "query": question,
            "limit": max_results,
            "filters": filters,
        }
    }
    print(f"  Tool: {tool_call['name']}")
    print(f"  Arguments: {json.dumps(tool_call['arguments'], indent=4)}")
    
    # Step 3: Tool Execution (Retrieval)
    print("\n[STEP 3] Tool Execution (Retrieval)")
    retrieved_recipes = recommender.search_recipes_tool(
        query=tool_call['arguments']['query'],
        limit=tool_call['arguments']['limit'],
        filters=tool_call['arguments']['filters'],
    )
    print(f"  ✅ Retrieved {len(retrieved_recipes)} recipes")
    for recipe in retrieved_recipes:
        print(f"     - [ID:{recipe['id']}] {recipe['title']}")
    
    # Step 4: Response Formatting (Grounded Generation)
    print("\n[STEP 4] Response Formatting")
    answer = format_response_with_citations(retrieved_recipes)
    print(f"  Generated response with {len(retrieved_recipes)} citations")
    
    # Step 5: Grounding Verification
    print("\n[STEP 5] Grounding Verification")
    verification = verify_response_grounding(answer, retrieved_recipes, question)
    print(f"  Grounded: {verification['grounded']}")
    print(f"  Verified IDs: {verification['verified_ids']}")
    if verification['issues']:
        print(f"  Issues: {verification['issues']}")
    else:
        print(f"  ✅ No issues - response is fully grounded!")
    
    # Step 6: Return Structured Response
    print("\n[STEP 6] Return Structured Response")
    final_response = {
        "question": question,
        "tool_call": tool_call,
        "retrieved_count": len(retrieved_recipes),
        "answer": answer,
        "verification": verification,
        "timestamps": {
            "executed_at": str(pd.Timestamp.now()),
        },
    }
    
    return final_response


# Test with multiple queries to show different scenarios
test_questions = [
    "high protein quick vegan recipes",
    "low carb keto friendly meals",
    "gluten-free vegan desserts"
]

all_responses = []
for question in test_questions:
    response = complete_agentic_pipeline(question, max_results=3)
    all_responses.append(response)
    
    print("\n" + "=" * 70)
    print("FINAL ANSWER")
    print("=" * 70)
    print(response["answer"])
    print()


## Section 9: Mirror the Complete Agentic Flow in the Notebook

This section demonstrates the complete end-to-end pipeline integrated with the backend.

**Flow**:
1. Load question
2. Preprocess query → Extract fuzzy filters
3. Execute search_recipes tool
4. Format response with citations (ONLY from retrieved results)
5. Verify grounding (detect hallucinations)
6. Return structured response with verification report

This mirrors the production flow in `backend/app/utils/agentic_rag.py::AgenticRecipeAssistant.answer()`

In [ ]:

import re

def verify_response_grounding(response_text: str, retrieved_recipes: list, question: str) -> dict:
    """
    Verify that the generated response is grounded in retrieved data.
    
    Returns:
        {
            "grounded": bool,  # True if all claims can be traced to retrieved data
            "hallucinated_ingredients": list,  # Ingredients mentioned but not retrieved
            "hallucinated_ids": list,  # Recipe IDs mentioned but not retrieved
            "verified_ids": list,  # IDs that were successfully cited
            "issues": list,  # Human-readable list of problems
        }
    """
    report = {
        "grounded": True,
        "hallucinated_ingredients": [],
        "hallucinated_ids": [],
        "verified_ids": [],
        "issues": [],
    }
    
    # Extract mentioned recipe IDs from response
    mentioned_ids = set()
    id_pattern = r'\[RID:(\d+)\]'
    for match in re.finditer(id_pattern, response_text):
        mentioned_ids.add(int(match.group(1)))
    
    # Verify all mentioned IDs are in retrieved results
    retrieved_ids = {int(r['id']) for r in retrieved_recipes}
    hallucinated_ids = mentioned_ids - retrieved_ids
    
    if hallucinated_ids:
        report["grounded"] = False
        report["hallucinated_ids"] = list(hallucinated_ids)
        report["issues"].append(f"❌ Hallucinated recipe IDs: {hallucinated_ids}")
        logger.error(f"Hallucinated IDs found: {hallucinated_ids}")
    else:
        report["verified_ids"] = list(mentioned_ids)
        logger.info(f"✅ All recipe IDs verified: {mentioned_ids}")
    
    # Extract ingredients mentioned in response
    mentioned_ingredients = set()
    for recipe in retrieved_recipes:
        ingredients_text = recipe.get('ingredients', '').lower()
        # Simple regex: look for ingredient names
        for ingredient in ingredients_text.split(','):
            mentioned_ingredients.add(ingredient.strip())
    
    # Check for potentially hallucinated ingredients in response text
    response_lower = response_text.lower()
    hallucinated_ingredients = []
    
    # Look for ingredient-like phrases NOT in the retrieved data
    common_ingredients_in_response = ['honey', 'milk', 'cheese', 'egg', 'meat', 'fish']
    for ing in common_ingredients_in_response:
        if ing in response_lower and ing not in ingredients_text.lower():
            hallucinated_ingredients.append(ing)
    
    if hallucinated_ingredients:
        report["grounded"] = False
        report["hallucinated_ingredients"] = hallucinated_ingredients
        report["issues"].append(f"❌ Potentially hallucinated ingredients: {hallucinated_ingredients}")
        logger.warning(f"Hallucinated ingredients detected: {hallucinated_ingredients}")
    else:
        logger.info("✅ No obviously hallucinated ingredients detected")
    
    return report


# Test the verification function
print("=" * 70)
print("Reliability Verification")
print("=" * 70)

verification = verify_response_grounding(
    formatted_response,
    agent_response["retrieved_results"],
    agent_response["question"]
)

print("\n📊 Verification Report:")
print(f"✅ Grounded: {verification['grounded']}")
print(f"📋 Verified IDs: {verification['verified_ids']}")
print(f"❌ Hallucinated IDs: {verification['hallucinated_ids']}")
print(f"❌ Hallucinated Ingredients: {verification['hallucinated_ingredients']}")
if verification["issues"]:
    print(f"\n🔍 Issues Found:")
    for issue in verification["issues"]:
        print(f"  {issue}")
else:
    print(f"\n✅ No issues found - response is fully grounded!")


## Section 8: Add Reliability Verification Against Retrieved Context

**Verification Pipeline**: Compare final output against retrieved tool context.

The verification function detects:
1. ❌ **Hallucinated Ingredients**: Mentioned ingredients NOT in retrieved recipes
2. ❌ **Fabricated Recipes**: Recipe IDs NOT in retrieved results
3. ❌ **False Nutrition Claims**: Protein/calorie values NOT from tool output
4. ✅ **Safe Claims**: All mentioned items traceable to retrieved data

If hallucinations detected → Log warning + Suggest correction

In [ ]:

def format_response_with_citations(retrieved_recipes: list) -> str:
    """
    Format the response STRICTLY using retrieved recipes.
    
    This enforces:
    1. Only recipes from tool output
    2. IDs are mandatory for citation
    3. No hallucinated ingredients
    """
    if not retrieved_recipes:
        return "❌ No recipes found matching your criteria. Try different keywords or relax constraints."
    
    response = f"🥬 **Found {len(retrieved_recipes)} Verified Vegan Recipes**\n\n"
    
    for recipe in retrieved_recipes:
        recipe_id = recipe.get('id')
        title = recipe.get('title', 'Unknown')
        ingredients = recipe.get('ingredients', 'N/A')
        protein = recipe.get('protein', 'N/A')
        calories = recipe.get('calories', 'N/A')
        prep_time = recipe.get('prep_time', 'N/A')
        cook_time = recipe.get('cook_time', 'N/A')
        
        # Mandatory citation format
        response += f"✅ **{title}** `[RID:{recipe_id}]`\n"
        response += f"   - **Protein:** {protein}g | **Calories:** {calories}\n"
        response += f"   - **Time:** {prep_time}min prep + {cook_time}min cook\n"
        response += f"   - **Ingredients:** {ingredients}\n"
        response += f"   - **Source:** Retrieved from database (ID: {recipe_id})\n\n"
    
    # Transparency footer
    response += "---\n"
    response += f"*All recipes retrieved from verified database and cited with IDs for transparency.*\n"
    response += f"*No ingredients were invented or assumed.*"
    
    return response


# Test the response formatter
test_results = agent_response["retrieved_results"]
formatted_response = format_response_with_citations(test_results)

print("=" * 70)
print("Formatted Response with Citations")
print("=" * 70)
print(formatted_response)


## Section 7: Generate Responses Strictly from Tool Results with Recipe ID Citations

Every response must:
1. **Only cite retrieved recipes**: Format: `[RID:<id>]`
2. **Include ingredient lists**: From tool output, never invented
3. **Provide nutritional info**: From tool output (protein, calories, etc.)
4. **Explain reasoning**: Why each recipe was selected or excluded

In [ ]:

print("=" * 70)
print("System Prompt from backend/app/prompts.py")
print("=" * 70)
print(SYSTEM_GROUNDED_PROMPT)
print("\n" + "=" * 70)
print("Verification Requirement Prompt")
print("=" * 70)
print(VERIFICATION_REQUIREMENT_PROMPT)


## Section 6: Strengthen Grounding Rules in `backend/prompts.py`

The **System Prompt** enforces strict reliability constraints on the LLM.

**Critical Rules**:
1. ✅ Only use information from `search_recipes` tool output
2. ✅ Citation requirement: Every recipe must include `[RID:<id>]`
3. ✅ No hallucination: Cannot invent recipes, ingredients, or nutrition values
4. ✅ Failure handling: If tool fails, inform user immediately
5. ✅ Transparency: Explain WHY a recipe was included or excluded

In [ ]:

def two_step_agent_loop(question: str, max_results: int = 5) -> dict:
    """
    Implement the Two-Step Agentic Loop.
    
    Step A: Tool Calling (Decision)
    Step B: Grounded Generation (Answer)
    """
    print("=" * 70)
    print(f"STEP A: Tool Calling (Planning)")
    print("=" * 70)
    print(f"User Question: '{question}'")
    print()
    
    # Step A: LLM decides to call search_recipes tool
    filters = fuzzy_query_to_filters(question)
    tool_call = {
        "tool": "search_recipes",
        "arguments": {
            "query": question,
            "limit": max_results,
            "filters": filters,
        },
    }
    
    print(f"🔧 Tool Call Decision:")
    print(f"   Tool: {tool_call['tool']}")
    print(f"   Query: {tool_call['arguments']['query']}")
    print(f"   Filters: {tool_call['arguments']['filters']}")
    print()
    
    # Execute the tool
    print("=" * 70)
    print("Tool Execution")
    print("=" * 70)
    results = recommender.search_recipes_tool(
        query=tool_call['arguments']['query'],
        limit=tool_call['arguments']['limit'],
        filters=tool_call['arguments']['filters'],
    )
    
    print(f"✅ Retrieved {len(results)} results from vector DB")
    for i, recipe in enumerate(results, 1):
        print(f"   {i}. [ID: {recipe['id']}] {recipe['title']}")
    print()
    
    # Step B: Generate answer using ONLY the tool results
    print("=" * 70)
    print("STEP B: Grounded Generation (Answer from Tool Results)")
    print("=" * 70)
    
    answer_text = f"Based on the search_recipes tool, I found {len(results)} vegan recipes:\n\n"
    for recipe in results:
        answer_text += f"✅ **{recipe['title']}** [RID:{recipe['id']}]\n"
        answer_text += f"   - Protein: {recipe.get('protein', 'N/A')}g | Calories: {recipe.get('calories', 'N/A')}\n"
        answer_text += f"   - Time: {recipe.get('prep_time', 0)}min prep + {recipe.get('cook_time', 0)}min cook\n"
        answer_text += f"   - Ingredients: {recipe.get('ingredients', 'N/A')[:60]}...\n\n"
    
    return {
        "question": question,
        "tool_call": tool_call,
        "retrieved_results": results,
        "answer": answer_text,
        "citations": [r['id'] for r in results],
    }


# Test the two-step agent loop
print("\n🎯 Testing Two-Step Agentic Agent Loop\n")
agent_response = two_step_agent_loop("high protein quick vegan recipes", max_results=3)

print("=" * 70)
print("FINAL RESPONSE")
print("=" * 70)
print(agent_response["answer"])
print(f"\nCitations: {agent_response['citations']}")


## Section 5: Add Agentic Tool-Call Orchestration

The **Two-Step Loop**:

1. **Step A (Query Planning)**: User asks question → LLM decides to call `search_recipes` tool
   - Tool call includes: query string + filters
   - LLM provides reasoning for why this tool is needed

2. **Step B (Grounded Generation)**: Tool returns results → LLM generates answer
   - LLM **must cite recipe IDs** in the response
   - LLM **cannot hallucinate** beyond retrieved results
   - System prompt enforces strict grounding

This is implemented in `backend/app/utils/agentic_rag.py` as the `AgenticRecipeAssistant` class.

In [ ]:

# Demonstrate the tool call and structured response
def execute_search_recipes_tool(query: str, limit: int = 5, filters: dict = None) -> list:
    """
    Execute the search_recipes tool.
    This is where the magic happens: semantic search + filtering.
    """
    if filters is None:
        filters = {}
    
    logger.info(f"🔧 Executing search_recipes tool")
    logger.info(f"   Query: '{query}'")
    logger.info(f"   Limit: {limit}")
    logger.info(f"   Filters: {filters}")
    
    # Call the backend tool
    results = recommender.search_recipes_tool(
        query=query,
        limit=limit,
        filters=filters
    )
    
    logger.info(f"✅ Tool returned {len(results)} recipes")
    return results


# Example: Execute the tool for a test query
print("=" * 70)
print("DEMO: Tool Execution with Structured Response")
print("=" * 70)

query = "high protein quick vegan recipes"
filters = fuzzy_query_to_filters(query)

results = execute_search_recipes_tool(query, limit=3, filters=filters)

print(f"\n📦 Tool Response (Grounding Context):\n")
for recipe in results:
    print(f"[ID: {recipe.get('id')}] {recipe.get('title')}")
    print(f"  Protein: {recipe.get('protein', 'N/A')}g | Calories: {recipe.get('calories', 'N/A')}")
    print(f"  Prep: {recipe.get('prep_time', 'N/A')}min | Cook: {recipe.get('cook_time', 'N/A')}min")
    print(f"  Ingredients: {recipe.get('ingredients', 'N/A')[:60]}...")
    print()


## Section 4: Refactor Retrieval Tool to Return Structured Grounding Context

The tool must return **stable, machine-readable data** that the LLM can reliably cite.

**Response Structure** (always includes these fields):
```json
{
  "id": 42,
  "title": "Tofu Scramble",
  "ingredients": "tofu, turmeric, nutritional yeast, spinach",
  "protein": 18.5,
  "calories": 250,
  "prep_time": 5,
  "cook_time": 10,
  "dietary_tags": "vegan,breakfast,high-protein"
}
```

**Key Requirements**:
1. **Unique IDs**: Every recipe must have a stable `id` for citation
2. **Complete Fields**: Always include required fields (never null)
3. **Standardized Formats**: Consistent types (string, float, int)
4. **Traceable Source**: Can audit which database record produced this output

In [ ]:

def fuzzy_query_to_filters(question: str) -> dict:
    """
    Map natural language query to structured filters.
    This is the DATA IMPEDANCE layer that constrains LLM behavior.
    """
    q = question.lower()
    filters = {
        "min_protein": None,
        "max_calories": None,
        "max_carbs": None,
        "max_total_time": None,
        "dietary_tags": [],
    }

    # Protein constraints
    if "high protein" in q or "protein-rich" in q or "protein packed" in q:
        filters["min_protein"] = 20
        logger.info("🔍 Filter: min_protein=20 (matched 'high protein')")
    
    # Carb constraints
    if "low carb" in q or "keto" in q or "carb-conscious" in q:
        filters["max_carbs"] = 20
        filters["dietary_tags"].append("keto")
        logger.info("🔍 Filter: max_carbs=20 (matched 'low carb')")
    
    # Calorie constraints
    if "low calorie" in q or "light" in q or "under 500" in q:
        filters["max_calories"] = 450
        logger.info("🔍 Filter: max_calories=450 (matched 'light' or 'low calorie')")
    
    # Time constraints
    if "quick" in q or "fast" in q or "under 30" in q or "30 min" in q:
        filters["max_total_time"] = 30
        logger.info("🔍 Filter: max_total_time=30 (matched 'quick' or 'fast')")
    
    # Dietary constraints
    if "gluten free" in q or "gluten-free" in q:
        filters["dietary_tags"].append("gluten-free")
        logger.info("🔍 Filter: gluten-free added")
    
    if "nut free" in q or "nut-free" in q:
        filters["dietary_tags"].append("nut-free")
        logger.info("🔍 Filter: nut-free added")

    return filters


# Test the fuzzy preprocessor with example queries
test_queries = [
    "high protein quick vegan recipes",
    "low carb keto meals",
    "light gluten-free desserts under 500 calories",
    "nut-free quick protein shakes"
]

print("🧪 Testing Fuzzy Query Preprocessor\n")
for query in test_queries:
    filters = fuzzy_query_to_filters(query)
    print(f"Query: '{query}'")
    print(f"Filters: {json.dumps({k: v for k, v in filters.items() if v}, indent=2)}\n")


## Section 3: Implement Fuzzy Query-to-Filter Preprocessing

**Data Impedance Problem**: The LLM generates natural language queries, but the vector DB needs structured filters.

**Solution**: Fuzzy matcher that maps common phrases to concrete constraints.

Example mappings:
- "high protein" → `min_protein: 20`
- "low carb" / "keto" → `max_carbs: 20`
- "quick" / "under 30 minutes" → `max_total_time: 30`
- "light meal" → `max_calories: 450`
- "gluten free" → `dietary_tags: ['gluten-free']`

This preprocessing happens **before** the tool call to ensure consistent behavior.

In [ ]:

# Define the search_recipes tool schema as JSON (OpenAPI-like format)
search_recipes_tool = {
    "type": "function",
    "function": {
        "name": "search_recipes",
        "description": "Retrieve vegan recipes from the database with semantic search and optional filters.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Natural language description (e.g., 'high protein quick meals', 'low carb desserts')"
                },
                "limit": {
                    "type": "integer",
                    "description": "Maximum number of recipes to return",
                    "default": 5,
                    "minimum": 1,
                    "maximum": 20
                },
                "filters": {
                    "type": "object",
                    "description": "Optional structured filters",
                    "properties": {
                        "min_protein": {"type": "number", "description": "Minimum protein in grams"},
                        "max_calories": {"type": "number", "description": "Maximum calories per serving"},
                        "max_carbs": {"type": "number", "description": "Maximum carbs in grams"},
                        "max_total_time": {"type": "number", "description": "Maximum total time in minutes"},
                        "dietary_tags": {
                            "type": "array",
                            "items": {"type": "string"},
                            "description": "Dietary restrictions (e.g., ['gluten-free', 'nut-free'])"
                        }
                    }
                }
            },
            "required": ["query"]
        }
    }
}

print("📋 Tool Schema: search_recipes")
print(json.dumps(search_recipes_tool, indent=2))


## Section 2: Define Tool Schema for `search_recipes` Function Calling

The agent uses a structured tool definition that constrains how the LLM requests recipe data.

**Tool Definition**: `search_recipes`
- **Purpose**: Retrieve recipes from the vector database with semantic search
- **Parameters**:
  - `query` (string, required): Natural language description of desired recipes
  - `limit` (int, optional): Max number of results (default: 5)
  - `filters` (dict, optional): Structured constraints (protein, calories, time, etc.)

**Response Format**: Always returns a list of dictionaries with:
- `id` (int): Unique recipe identifier for citation
- `title` (str): Recipe name
- `ingredients` (str): Ingredient list
- `protein` (float): Protein in grams
- `calories` (float): Calories per serving
- `prep_time` (float): Preparation time in minutes
- `cook_time` (float): Cooking time in minutes

In [ ]:

# Load the recipe dataset and initialize the recommender
import pandas as pd

# Try multiple possible paths for the CSV
csv_paths = [
    "../vegan_recipes.csv",
    "../HuggingFaceSpaces/vegan_recipes.csv",
    "../backend/vegan_recipes.csv",
]

csv_path = None
for path in csv_paths:
    if os.path.exists(path):
        csv_path = path
        break

if csv_path:
    print(f"✅ Found recipes CSV at: {csv_path}")
    recommender = RecipeRecommender(csv_path)
    agentic_assistant = AgenticRecipeAssistant(recommender)
    print(f"✅ RecipeRecommender initialized with {len(recommender.df)} recipes")
    print(f"✅ AgenticRecipeAssistant with verification agent ready")
else:
    print("❌ CSV file not found at expected paths")
    print(f"   Checked: {csv_paths}")


In [ ]:

import sys
import os
import json
import logging
from pathlib import Path

# Add backend to path
backend_path = Path("../backend")
sys.path.insert(0, str(backend_path.resolve()))

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("agentic_rag_notebook")

# Import backend modules
from app.utils.recommend import RecipeRecommender
from app.utils.agentic_rag import AgenticRecipeAssistant
from app.services.data_fetcher import get_verifier, ANIMAL_DERIVED_INGREDIENTS
from app.prompts import SYSTEM_GROUNDED_PROMPT, VERIFICATION_REQUIREMENT_PROMPT

print("✅ Backend modules imported successfully")
print(f"Backend path: {backend_path.resolve()}")


## Section 1: Load Current Backend Components and Trace Query Flow

We start by importing the refactored backend modules and understanding the query pipeline.

**Key Files**:
- `backend/app/main.py` - FastAPI server with agentic query endpoint
- `backend/app/services/data_fetcher.py` - External API integrations (USDA, Open Food Facts)
- `backend/app/services/verification_agent.py` - Two-step verification orchestration
- `backend/app/utils/agentic_rag.py` - Agentic RAG assistant with tool calling
- `backend/app/prompts.py` - Grounding constraints and system prompts

# 🥬 Veg Vibe: Agentic RAG with Two-Step Verification
## Senior AI Engineer - Backend Refactoring

This notebook demonstrates the **Agentic Interface** for Veg Vibe, implementing:
- ✅ **Tool-Use Logic**: Structured `search_recipes` function calling
- ✅ **Grounding Constraints**: System prompts that enforce evidence-based reasoning
- ✅ **Data Impedance**: Natural language → Structured filters mapping
- ✅ **Reliability Verification**: Hallucination detection & correction

### Architecture Overview

```
Step A: Query
User: "high protein quick vegan recipes"
  ↓
[search_recipes Tool] (Fuzzy filters + semantic search)
  ↓
Retrieved Results: [{id, title, ingredients, protein, ...}]

Step B: Verify
Retrieved Results
  ↓
[Verification Tool] (Check against USDA, Open Food Facts, PETA list)
  ↓
Verified Results: Only safe recipes returned
```

---